##  Bước 0: Mount Google Drive

Kết nối với Google Drive để lưu/load models

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Tạo thư mục lưu models (nếu chưa có)
DRIVE_BASE = '/content/drive/MyDrive/RAG_Backend_Cache'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/ollama_models', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/bge_m3_cache', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/backend_code', exist_ok=True)

print("Google Drive đã được mount tại:", DRIVE_BASE)
print("Cấu trúc thư mục:")
print("   ", f"{DRIVE_BASE}/ollama_models/    (Lưu Qwen2.5:7b)")
print("   ", f"{DRIVE_BASE}/bge_m3_cache/      (Lưu BGE-M3)")
print("   ", f"{DRIVE_BASE}/backend_code/      (Lưu backend.zip)")

---
##  Bước 1: Cài đặt Ollama

Cài đặt Ollama và dependencies

In [ ]:
# Cài đặt dependencies
!sudo apt-get update -qq && sudo apt-get install -y zstd pciutils

# Cài đặt Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("Đã cài đặt Ollama")

In [ ]:
# Khởi động Ollama server
import subprocess
import time

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Đợi Ollama khởi động...")
time.sleep(5)
print("Ollama đã sẵn sàng")

---
## Bước 2: Load Model Qwen2.5-7b




In [ ]:
import os
import shutil

DRIVE_MODEL_PATH = f'{DRIVE_BASE}/ollama_models/qwen2.5_7b'
LOCAL_OLLAMA_MODELS = os.path.expanduser('~/.ollama/models')

# Kiểm tra xem Drive đã có model chưa
if os.path.exists(DRIVE_MODEL_PATH) and os.listdir(DRIVE_MODEL_PATH):
    print("Tìm thấy model trong Drive!")
    print("Đang copy từ Drive → Local...")

    # Copy từ Drive xuống local
    if os.path.exists(LOCAL_OLLAMA_MODELS):
        shutil.rmtree(LOCAL_OLLAMA_MODELS)

    shutil.copytree(DRIVE_MODEL_PATH, LOCAL_OLLAMA_MODELS)
    print("Đã restore model từ Drive (tiết kiệm ~5-10 phút!)")

else:
    print("Chưa có model trong Drive, đang tải về...")
    print("Sẽ mất 5-10 phút tùy tốc độ mạng")

    # Pull model từ Ollama
    !ollama pull qwen2.5:7b

    print("\n Đang lưu model vào Drive cho lần sau...")

    # Backup vào Drive
    if os.path.exists(DRIVE_MODEL_PATH):
        shutil.rmtree(DRIVE_MODEL_PATH)

    shutil.copytree(LOCAL_OLLAMA_MODELS, DRIVE_MODEL_PATH)
    print("Đã lưu model vào Drive!")

# Kiểm tra model
print("\n Danh sách models:")
!ollama list

In [ ]:
# Test Ollama
import requests

try:
    r = requests.get("http://localhost:11434/api/tags")
    print(" Ollama server online")
    print(" Models:", r.json())

    # Test chat
    response = requests.post("http://localhost:11434/api/chat", json={
        "model": "qwen2.5:7b",
        "messages": [{"role": "user", "content": "Xin chào!"}],
        "stream": False
    })

    print("\n Test response:")
    print(response.json()["message"]["content"])

except Exception as e:
    print("Lỗi:", e)

---
##  Bước 3: Upload Backend Code



In [ ]:
import os
import shutil
from google.colab import files

DRIVE_BACKEND_ZIP = f'{DRIVE_BASE}/backend_code/backend.zip'

# Kiểm tra xem Drive đã có backend.zip chưa
if os.path.exists(DRIVE_BACKEND_ZIP):
    print("Tìm thấy backend.zip trong Drive")

    use_drive = input("Dùng backend từ Drive? (y/n): ").lower().strip()

    if use_drive == 'y':
        shutil.copy(DRIVE_BACKEND_ZIP, '/content/backend.zip')
        print("Đã copy backend.zip từ Drive")
    else:
        print(" Upload backend.zip mới từ máy tính...")
        uploaded = files.upload()
        # Lưu vào Drive
        shutil.copy('/content/backend.zip', DRIVE_BACKEND_ZIP)
        print("Đã lưu backend.zip vào Drive")
else:
    print(" Upload backend.zip từ máy tính...")
    uploaded = files.upload()
    # Lưu vào Drive
    shutil.copy('/content/backend.zip', DRIVE_BACKEND_ZIP)
    print(" Đã lưu backend.zip vào Drive")

In [ ]:
# Giải nén backend.zip
import zipfile
import os

with zipfile.ZipFile("/content/backend.zip", "r") as z:
    z.extractall("/content/")

print("Đã giải nén vào /content/backend")
print("Nội dung:", os.listdir("/content/backend"))

---
##  Bước 4: Cài đặt Dependencies

In [ ]:
%cd /content/backend
!pip install -q -r requirements.txt
print(" Đã cài đặt xong dependencies")

---
##  Bước 5: Load BGE-M3 Model (Smart Cache)


In [ ]:
import os
import shutil
from pathlib import Path

DRIVE_BGE_CACHE = f'{DRIVE_BASE}/bge_m3_cache'
LOCAL_HF_CACHE = os.path.expanduser('~/.cache/huggingface')

# Kiểm tra xem Drive đã có BGE-M3 cache chưa
bge_model_path = f'{DRIVE_BGE_CACHE}/hub/models--BAAI--bge-m3'

if os.path.exists(bge_model_path):
    print("Tìm thấy BGE-M3 cache trong Drive!")
    print(" Đang tạo symlink để dùng trực tiếp từ Drive...")

    # Tạo symlink từ local cache → Drive cache
    os.makedirs(os.path.dirname(LOCAL_HF_CACHE), exist_ok=True)

    if os.path.exists(LOCAL_HF_CACHE):
        if os.path.islink(LOCAL_HF_CACHE):
            os.unlink(LOCAL_HF_CACHE)
        else:
            shutil.rmtree(LOCAL_HF_CACHE)

    os.symlink(DRIVE_BGE_CACHE, LOCAL_HF_CACHE)
    print(" Đã link cache từ Drive (tiết kiệm thời gian!)")

else:
    print("Chưa có BGE-M3 trong Drive, đang tải về...")
    print("Sẽ mất 2-3 phút")

    # Load model (sẽ tự động tải về local cache)
    from FlagEmbedding import BGEM3FlagModel
    model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
    del model

    print("\n Đang copy cache vào Drive cho lần sau...")

    # Copy cache vào Drive
    if os.path.exists(LOCAL_HF_CACHE):
        shutil.copytree(LOCAL_HF_CACHE, DRIVE_BGE_CACHE, dirs_exist_ok=True)
        print(" Đã lưu BGE-M3 cache vào Drive!")

# Test load model
print("\n Test load BGE-M3...")
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print(" BGE-M3 đã sẵn sàng")
del model

---
##  Bước 6: Khởi động Backend

In [ ]:
# Cell 1: Kill backend cũ
import subprocess, time, requests

# Kill tat ca process uvicorn dang chay
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(3)

# Kiem tra port da duoc giai phong chua
try:
    requests.get("http://localhost:8000/health", timeout=2)
    print("Port 8000 van con process - thu lai")
except:
    print("Port 8000 da trong, co the khoi dong backend moi")

In [ ]:
import subprocess
import time
import requests

# Khởi động backend
proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("Chờ backend khởi động...")
backend_ready = False

for i in range(12):
    time.sleep(5)
    try:
        r = requests.get("http://localhost:8000/health", timeout=5)
        print(" Backend đã online!")
        print(" Health check:", r.json())
        backend_ready = True
        break
    except:
        print(f"   [{(i+1)*5}s] Đang chờ...")

if not backend_ready:
    print(" Backend không khởi động được")
    print(" Log:")
    print(proc.stdout.read(3000).decode())

---
##  Bước 7: Tạo Public URL

In [ ]:
# Cài đặt pyngrok
!pip install -q pyngrok

print(" Đã cài đặt pyngrok")

In [ ]:
from pyngrok import ngrok


# Tao tunnel moi
ngrok.set_auth_token("3Ao1gQVME1XKXxsq9IYxQb2Stkd_7oEuvMqpXRghaEFtsPbf")
tunnel = ngrok.connect(8000)
print("BACKEND URL:", tunnel.public_url)